# Independent MSL GaLS elasticity fields

This notebook replays the seven archived **MSL_MHM + MSL_CG (GaLS)** and pyMHM
comparisons. It validates public archive checksums and the recorded field-norm
bounds, then displays the comparison figures. It does not rerun either solver.

The matched unit-square problem has `mu=1`, `lambda=4999` (`nu=.4999`), zero
boundary displacement and the consistent amplitude-one trigonometric data.
Herrmann pressure is `p=-lambda*div(u)`, and Cauchy stress is
`sigma=2*mu*sym(grad(u))-p*I`. The skeleton represents `-sigma @ n`.

Five P1/P1 cases refine a crisscross macro mesh; two additional cases use P2/P2
and P3/P3. Every macroface has one discontinuous linear vector trace segment.
These are matched code comparisons, not reproductions of the six skeletal
refinement tables in arXiv:2403.16890v1.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import SVG, display

record_path = root / "examples/results/elasticity-reference.json"
record = json.loads(record_path.read_text())
rows = record["rows"]
assert len(rows) == 7
for row in rows:
    archive = record_path.parent / row["archive"]
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
    with np.load(archive, allow_pickle=False) as arrays:
        for key in arrays.files:
            assert np.all(np.isfinite(arrays[key])), (archive.name, key)
        assert int(arrays["degree"]) == row["degree"]
        assert len(arrays["macro_cells"]) == row["macro_triangles"]
print(record["reference"])
print(record["revisions"])
print("Seven archived field checksums and array contracts passed.")


## Agreement and accuracy are different diagnostics

The stored `difference_l2` values compare both reconstructed polynomial fields
on the same physical quadrature. They include displacement, its full broken
gradient, pressure and full Cauchy stress. No pressure-mean shift was applied.

The checks below bound the maximum recorded implementation difference and
verify the reverse triangle inequality against the two approximation errors.
They inspect recorded norms; their scale is not a discretization convergence
rate, and agreement between codes does not by itself establish exact-solution
accuracy.


In [ ]:
limits = {"u_l2": 3e-13, "p_l2": 9e-12,
          "u_h1_semi": 2e-11, "stress_l2": 4e-11}
maxima = {metric: max(row["difference_l2"][metric] for row in rows) for metric in limits}
for metric, limit in limits.items():
    assert maxima[metric] <= limit
    for row in rows:
        first, second = row["msl"][metric], row["pymhm"][metric]
        rounding = 64*np.finfo(float).eps*(1+abs(first)+abs(second))
        assert abs(first-second) <= row["difference_l2"][metric] + rounding
maxima


In [ ]:
display(SVG(filename=str(root / "docs/figures/elasticity-reference/field-differences.svg")))


## Five matched mesh levels and two additional local pairs

A crisscross grid with `n` Cartesian squares per side has `4*n**2` macrotriangles
and macro diameter `H=1/n`. P1/P1 uses four fine subdivisions per macro edge and
stabilization `alpha=.1`, at `n=1,2,4,8,16`. The P2/P2 and P3/P3 checks both use
`n=4` and `alpha=.001`, with local refinements two and one respectively.

Assembly requested quadrature order 10, and field comparison order 12. Each
implementation may realize its assembly order using a different quadrature
rule. The additional-degree checks change the fine mesh too, so they are not
a fixed-mesh polynomial-rate experiment.


In [ ]:
for row in rows:
    print(f"n={row['macro_resolution']:2}, degree={row['degree']}, "
          f"local refinement={row['local_refinement']}: {row['pymhm']}")
display(SVG(filename=str(root / "docs/figures/elasticity-reference/convergence.svg")))


## Exact, MSL and pyMHM fields on the actual macro mesh

The maps show the P3/P3 case at `n=4`. The analytical, MSL and pyMHM fields share
one scale. The two approximation-error maps share another scale, while the
implementation difference has its own labeled colorbar. Macro boundaries appear
in every panel. Each broken fine-element polynomial is evaluated separately;
one-sided stress and pressure values are not smoothed across interfaces.

The displacement-pressure GaLS stress is symmetric. This reconstruction does
not enforce global H(div) conformity of stress.


In [ ]:
for field in ("u1", "pressure", "stress11"):
    display(SVG(filename=str(root / f"docs/figures/elasticity-reference/{field}.svg")))


## Recorded reference provenance

The report identifies the exact MSL_MHM, MSL_CG and MSL_CORE revisions and the
mixed-field adapter digests. Native GaLS element assembly was retained. The
adapter pads rigid-mode source projection with zero pressure rows and
reconstructs pressure using the same native skeletal coefficients as
displacement. This is a recorded adapter around the native operators, not an
assertion that an unchanged historical application generated the files.

The consistent amplitude-one benchmark matches the MSL input. It does not mix
the incompatible coefficients printed for displacement, pressure and forcing
in the article. The full derivation and scope are in
`docs/cases/elasticity.md` and `docs/cases/elasticity-reference.md`.

To redraw these archived fields, run
`pixi run -e notebooks python examples/plot_elasticity_reference.py`.
